# Generer GitHub Pages Dashboard

Denne notebooken henter ferske data fra Gold-tabellene og genererer en statisk HTML-fil
som kan publiseres på GitHub Pages som en visuell showcase av Pensjon Lakehouse.

**Flyt:**
1. Kjør SQL-spørringer mot Gold-tabellene i Unity Catalog
2. Samle data som Python-dicts
3. Generer `index.html` med Chart.js-visualiseringer
4. Last ned filen og push til GitHub

**Forutsetning:** `01_bronze_ingest`, `02_silver` og `03_gold` er kjørt.

## 1. Hent data fra Gold-tabellene

In [0]:
import json
from datetime import datetime

# --- KPI-er og pensjonsandel-trend ---
df_trend = spark.sql("""
    SELECT
        CAST(year AS INT) AS year,
        pensjonsandel_pst,
        total_55_pluss,
        total_befolkning
    FROM pensjon_lakehouse.gold.pensjonsandel_trend
    ORDER BY year
""").toPandas()

# --- Aldersfordeling siste år ---
df_alder = spark.sql("""
    SELECT
        aldersgruppe,
        aldersgruppe_sortering,
        befolkning,
        ROUND(andel * 100, 1) AS andel_pst
    FROM pensjon_lakehouse.gold.aldersgruppe_fordeling
    ORDER BY aldersgruppe_sortering
""").toPandas()

# --- Top 10 kommuner ---
df_kommuner = spark.sql("""
    SELECT
        kommune_label AS kommune,
        total_befolkning AS innbyggere,
        pension_age_befolkning AS innbyggere_55_pluss,
        ROUND(pension_age_share * 100, 1) AS andel_pst
    FROM pensjon_lakehouse.gold.top_kommuner_pensjonsalder
    ORDER BY andel_pst DESC
    LIMIT 10
""").toPandas()

# --- Top 10 næringer ---
df_naering = spark.sql("""
    SELECT
        naering_label AS naering,
        lonsstakere,
        manedslonn,
        ROUND(estimert_pensjonsvolum / 1e9, 2) AS volum_mrd
    FROM pensjon_lakehouse.gold.naering_pensjonsvolum
    WHERE naering_label != 'Alle næringer'
    ORDER BY volum_mrd DESC
    LIMIT 10
""").toPandas()

# --- Aldersgruppe-trend over tid ---
df_aldertrend = spark.sql("""
    SELECT
        CAST(year AS INT) AS year,
        aldersgruppe,
        aldersgruppe_sortering,
        ROUND(andel * 100, 1) AS andel_pst
    FROM pensjon_lakehouse.gold.aldersgruppe_trend
    ORDER BY year, aldersgruppe_sortering
""").toPandas()

# --- Detailtabell: alle kommuner siste år ---
df_alle_kommuner = spark.sql("""
    SELECT
        kommune_label AS kommune,
        total_befolkning AS innbyggere,
        pension_age_befolkning AS innbyggere_55_pluss,
        ROUND(pension_age_share * 100, 1) AS andel_pst
    FROM pensjon_lakehouse.silver.befolkning_pensjon
    WHERE year = (SELECT MAX(year) FROM pensjon_lakehouse.silver.befolkning_pensjon)
    ORDER BY andel_pst DESC
""").toPandas()

# KPI-verdier fra siste år
latest = df_trend.iloc[-1]
kpi_pensjonsandel = float(latest['pensjonsandel_pst'])
kpi_55_pluss = int(latest['total_55_pluss'])
kpi_total = int(latest['total_befolkning'])
kpi_year = int(latest['year'])

print(f"✓ Data hentet ({kpi_year})")
print(f"  Pensjonsandel: {kpi_pensjonsandel}%")
print(f"  55+: {kpi_55_pluss:,}")
print(f"  Total: {kpi_total:,}")

## 2. Konverter til JSON for HTML-templaten

In [0]:
# Pensjonsandel-trend
data_trend = {
    "years": df_trend['year'].tolist(),
    "values": df_trend['pensjonsandel_pst'].round(1).tolist()
}

# Aldersfordeling
data_alder = {
    "labels": df_alder['aldersgruppe'].tolist(),
    "values": df_alder['befolkning'].tolist(),
    "andel": df_alder['andel_pst'].tolist()
}

# Top kommuner
data_kommuner = {
    "labels": df_kommuner['kommune'].tolist(),
    "values": df_kommuner['andel_pst'].tolist()
}

# Top næringer
data_naering = {
    "labels": df_naering['naering'].tolist(),
    "values": df_naering['volum_mrd'].tolist()
}

# Aldersgruppe-trend (pivoter til {gruppe: [verdier per år]})
grupper = df_aldertrend.sort_values('aldersgruppe_sortering')['aldersgruppe'].unique().tolist()
trend_years = sorted(df_aldertrend['year'].unique().tolist())
trend_series = {}
for g in grupper:
    subset = df_aldertrend[df_aldertrend['aldersgruppe'] == g].set_index('year')
    trend_series[g] = [float(subset.loc[y, 'andel_pst']) if y in subset.index else 0 for y in trend_years]

data_aldertrend = {
    "years": trend_years,
    "series": trend_series
}

# Detailtabell
data_tabell = df_alle_kommuner.to_dict(orient='records')

print(f"✓ JSON-data klar")
print(f"  Trend: {len(data_trend['years'])} år")
print(f"  Aldersgrupper: {len(data_alder['labels'])}")
print(f"  Kommuner (top): {len(data_kommuner['labels'])}")
print(f"  Næringer (top): {len(data_naering['labels'])}")
print(f"  Kommuner (alle): {len(data_tabell)}")

## 3. Generer HTML-dashboard

In [0]:
generated_date = datetime.now().strftime("%Y-%m-%d %H:%M")

html = f"""<!DOCTYPE html>
<html lang="no">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Pensjon Lakehouse Dashboard</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.7/dist/chart.umd.min.js"></script>
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;700&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
    <style>
        :root {{
            --bg-canvas: #0F172A;
            --bg-widget: #111827;
            --border: #334155;
            --text: #E5E7EB;
            --text-dim: #94A3B8;
            --accent: #38BDF8;
            --green: #22C55E;
            --amber: #F59E0B;
            --purple: #A78BFA;
            --rose: #F43F5E;
            --teal: #14B8A6;
            --orange: #F97316;
            --yellow: #EAB308;
        }}

        * {{ margin: 0; padding: 0; box-sizing: border-box; }}

        body {{
            font-family: 'DM Sans', sans-serif;
            background: var(--bg-canvas);
            color: var(--text);
            min-height: 100vh;
            padding: 24px;
        }}

        .dashboard {{
            max-width: 1280px;
            margin: 0 auto;
        }}

        .header {{
            margin-bottom: 32px;
            padding-bottom: 20px;
            border-bottom: 1px solid var(--border);
        }}

        .header h1 {{
            font-size: 28px;
            font-weight: 700;
            letter-spacing: -0.5px;
            margin-bottom: 6px;
        }}

        .header h1 span {{
            color: var(--accent);
        }}

        .header p {{
            color: var(--text-dim);
            font-size: 14px;
        }}

        .header p a {{
            color: var(--accent);
            text-decoration: none;
        }}

        .header p a:hover {{
            text-decoration: underline;
        }}

        .kpi-row {{
            display: grid;
            grid-template-columns: repeat(3, 1fr);
            gap: 16px;
            margin-bottom: 16px;
        }}

        .kpi-card {{
            background: var(--bg-widget);
            border: 1px solid var(--border);
            border-radius: 10px;
            padding: 20px 24px;
        }}

        .kpi-card .label {{
            font-size: 12px;
            font-weight: 500;
            color: var(--text-dim);
            text-transform: uppercase;
            letter-spacing: 0.8px;
            margin-bottom: 6px;
        }}

        .kpi-card .value {{
            font-family: 'JetBrains Mono', monospace;
            font-size: 32px;
            font-weight: 500;
            color: var(--accent);
        }}

        .grid-2 {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 16px;
            margin-bottom: 16px;
        }}

        .grid-1 {{
            margin-bottom: 16px;
        }}

        .widget {{
            background: var(--bg-widget);
            border: 1px solid var(--border);
            border-radius: 10px;
            padding: 20px 24px;
        }}

        .widget h2 {{
            font-size: 14px;
            font-weight: 500;
            color: var(--text-dim);
            margin-bottom: 16px;
        }}

        .chart-container {{
            position: relative;
            height: 300px;
        }}

        .chart-container-wide {{
            position: relative;
            height: 350px;
        }}

        /* Tabell */
        .table-wrapper {{
            overflow-x: auto;
            max-height: 480px;
            overflow-y: auto;
        }}

        table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
        }}

        thead {{
            position: sticky;
            top: 0;
            z-index: 1;
        }}

        th {{
            background: #1E293B;
            color: var(--text-dim);
            font-weight: 500;
            text-align: left;
            padding: 10px 14px;
            border-bottom: 1px solid var(--border);
            cursor: pointer;
            user-select: none;
            white-space: nowrap;
        }}

        th:hover {{
            color: var(--accent);
        }}

        th .sort-arrow {{
            margin-left: 4px;
            opacity: 0.4;
        }}

        th.sorted .sort-arrow {{
            opacity: 1;
            color: var(--accent);
        }}

        td {{
            padding: 8px 14px;
            border-bottom: 1px solid rgba(51, 65, 85, 0.4);
        }}

        tr:hover td {{
            background: rgba(56, 189, 248, 0.04);
        }}

        .num {{
            font-family: 'JetBrains Mono', monospace;
            text-align: right;
            font-size: 12px;
        }}

        .andel-bar {{
            display: flex;
            align-items: center;
            gap: 8px;
        }}

        .andel-bar .bar {{
            height: 6px;
            border-radius: 3px;
            background: var(--accent);
            opacity: 0.6;
        }}

        .search-box {{
            margin-bottom: 12px;
        }}

        .search-box input {{
            background: #1E293B;
            border: 1px solid var(--border);
            color: var(--text);
            padding: 8px 14px;
            border-radius: 6px;
            font-family: 'DM Sans', sans-serif;
            font-size: 13px;
            width: 260px;
            outline: none;
        }}

        .search-box input:focus {{
            border-color: var(--accent);
        }}

        .search-box input::placeholder {{
            color: var(--text-dim);
        }}

        .footer {{
            text-align: center;
            padding: 32px 0 16px;
            color: var(--text-dim);
            font-size: 12px;
        }}

        .footer a {{
            color: var(--accent);
            text-decoration: none;
        }}

        @media (max-width: 768px) {{
            .kpi-row {{ grid-template-columns: 1fr; }}
            .grid-2 {{ grid-template-columns: 1fr; }}
            .kpi-card .value {{ font-size: 24px; }}
            body {{ padding: 12px; }}
        }}
    </style>
</head>
<body>

<div class="dashboard">

    <div class="header">
        <h1>Pensjon <span>Lakehouse</span></h1>
        <p>Pensjonsdemografi i Norge &middot; SSB 07459 + 11654 &middot; Bronze &rarr; Silver &rarr; Gold &middot;
           <a href="https://github.com/FredrikVE/Pensjon-Lakehouse" target="_blank">GitHub</a></p>
    </div>

    <!-- KPI-er -->
    <div class="kpi-row">
        <div class="kpi-card">
            <div class="label">Pensjonsandel 55+ ({kpi_year})</div>
            <div class="value">{kpi_pensjonsandel}%</div>
        </div>
        <div class="kpi-card">
            <div class="label">Innbyggere 55+</div>
            <div class="value">{kpi_55_pluss:,}</div>
        </div>
        <div class="kpi-card">
            <div class="label">Total befolkning</div>
            <div class="value">{kpi_total:,}</div>
        </div>
    </div>

    <!-- Rad 2: Trend + Aldersfordeling -->
    <div class="grid-2">
        <div class="widget">
            <h2>Pensjonsandel over tid</h2>
            <div class="chart-container"><canvas id="chartTrend"></canvas></div>
        </div>
        <div class="widget">
            <h2>Aldersfordeling</h2>
            <div class="chart-container"><canvas id="chartAlder"></canvas></div>
        </div>
    </div>

    <!-- Rad 3: Kommuner + Næringer -->
    <div class="grid-2">
        <div class="widget">
            <h2>Top 10 kommuner &mdash; andel 55+</h2>
            <div class="chart-container"><canvas id="chartKommuner"></canvas></div>
        </div>
        <div class="widget">
            <h2>Top 10 n&aelig;ringer &mdash; pensjonsvolum (mrd kr)</h2>
            <div class="chart-container"><canvas id="chartNaering"></canvas></div>
        </div>
    </div>

    <!-- Rad 4: Stacked area -->
    <div class="grid-1">
        <div class="widget">
            <h2>Aldersgrupper over tid</h2>
            <div class="chart-container-wide"><canvas id="chartAlderTrend"></canvas></div>
        </div>
    </div>

    <!-- Rad 5: Tabell -->
    <div class="grid-1">
        <div class="widget">
            <h2>Kommune-detaljer</h2>
            <div class="search-box">
                <input type="text" id="tableSearch" placeholder="S&oslash;k kommune..." oninput="filterTable()">
            </div>
            <div class="table-wrapper">
                <table id="kommuneTable">
                    <thead>
                        <tr>
                            <th onclick="sortTable(0)">Kommune <span class="sort-arrow">&udarr;</span></th>
                            <th onclick="sortTable(1)" class="num">Innbyggere <span class="sort-arrow">&udarr;</span></th>
                            <th onclick="sortTable(2)" class="num">55+ <span class="sort-arrow">&udarr;</span></th>
                            <th onclick="sortTable(3)" class="num">Andel 55+ <span class="sort-arrow">&udarr;</span></th>
                        </tr>
                    </thead>
                    <tbody id="tableBody"></tbody>
                </table>
            </div>
        </div>
    </div>

    <div class="footer">
        Generert {generated_date} &middot; Data: <a href="https://www.ssb.no/statbank/table/07459" target="_blank">SSB 07459</a> +
        <a href="https://www.ssb.no/statbank/table/11654" target="_blank">SSB 11654</a> &middot;
        <a href="https://github.com/FredrikVE/Pensjon-Lakehouse" target="_blank">FredrikVE/Pensjon-Lakehouse</a>
    </div>
</div>

<script>
// ── Data ──
const dataTrend = {json.dumps(data_trend)};
const dataAlder = {json.dumps(data_alder, ensure_ascii=False)};
const dataKommuner = {json.dumps(data_kommuner, ensure_ascii=False)};
const dataNaering = {json.dumps(data_naering, ensure_ascii=False)};
const dataAlderTrend = {json.dumps(data_aldertrend, ensure_ascii=False)};
const dataTabell = {json.dumps(data_tabell, ensure_ascii=False)};

const COLORS = ['#38BDF8','#22C55E','#F59E0B','#A78BFA','#F43F5E','#14B8A6','#F97316','#EAB308'];

Chart.defaults.color = '#94A3B8';
Chart.defaults.borderColor = 'rgba(51,65,85,0.4)';
Chart.defaults.font.family = "'DM Sans', sans-serif";

// ── Pensjonsandel-trend (linje) ──
new Chart(document.getElementById('chartTrend'), {{
    type: 'line',
    data: {{
        labels: dataTrend.years,
        datasets: [{{
            data: dataTrend.values,
            borderColor: '#38BDF8',
            backgroundColor: 'rgba(56,189,248,0.1)',
            fill: true,
            tension: 0.3,
            pointRadius: 3,
            pointBackgroundColor: '#38BDF8',
            borderWidth: 2
        }}]
    }},
    options: {{
        responsive: true,
        maintainAspectRatio: false,
        plugins: {{ legend: {{ display: false }} }},
        scales: {{
        
            x: {{ grid: {{ display: false }} }},
            
            y: {{ 
                ticks: {{ 
                    precision: 1,
                    callback: v => Number(v).toFixed(1) + '%'
                }}
            }}
        }}
    }}
}});

// ── Aldersfordeling (søyler) ──
new Chart(document.getElementById('chartAlder'), {{
    type: 'bar',
    data: {{
        labels: dataAlder.labels,
        datasets: [{{
            data: dataAlder.values,
            backgroundColor: COLORS.slice(0, dataAlder.labels.length),
            borderRadius: 4
        }}]
    }},
    options: {{
        responsive: true,
        maintainAspectRatio: false,
        plugins: {{ legend: {{ display: false }} }},
        scales: {{
            x: {{ grid: {{ display: false }} }},
            y: {{ ticks: {{ callback: v => (v >= 1000000) ? (v/1000000).toFixed(1)+'M' : (v >= 1000) ? (v/1000).toFixed(0)+'k' : v }} }}
        }}
    }}
}});

// ── Top kommuner (horisontal) ──
new Chart(document.getElementById('chartKommuner'), {{
    type: 'bar',
    data: {{
        labels: dataKommuner.labels.slice().reverse(),
        datasets: [{{
            data: dataKommuner.values.slice().reverse(),
            backgroundColor: '#38BDF8',
            borderRadius: 4
        }}]
    }},
    options: {{
        indexAxis: 'y',
        responsive: true,
        maintainAspectRatio: false,
        plugins: {{ legend: {{ display: false }} }},
        scales: {{
            x: {{ ticks: {{ callback: v => v + '%' }} }},
            y: {{ grid: {{ display: false }} }}
        }}
    }}
}});

// ── Top næringer (horisontal) ──
new Chart(document.getElementById('chartNaering'), {{
    type: 'bar',
    data: {{
        labels: dataNaering.labels.slice().reverse(),
        datasets: [{{
            data: dataNaering.values.slice().reverse(),
            backgroundColor: '#22C55E',
            borderRadius: 4
        }}]
    }},
    options: {{
        indexAxis: 'y',
        responsive: true,
        maintainAspectRatio: false,
        plugins: {{ legend: {{ display: false }} }},
        scales: {{
            x: {{ ticks: {{ callback: v => v + ' mrd' }} }},
            y: {{ grid: {{ display: false }} }}
        }}
    }}
}});

// ── Aldersgrupper over tid (stacked area) ──
const alderGrupper = Object.keys(dataAlderTrend.series);
new Chart(document.getElementById('chartAlderTrend'), {{
    type: 'line',
    data: {{
        labels: dataAlderTrend.years,
        datasets: alderGrupper.map((g, i) => ({{
            label: g,
            data: dataAlderTrend.series[g],
            borderColor: COLORS[i % COLORS.length],
            backgroundColor: COLORS[i % COLORS.length] + '66',
            fill: true,
            tension: 0.3,
            pointRadius: 0,
            borderWidth: 1.5
        }}))
    }},
    options: {{
        responsive: true,
        maintainAspectRatio: false,
        plugins: {{
            legend: {{
                position: 'bottom',
                labels: {{ boxWidth: 12, padding: 16, font: {{ size: 11 }} }}
            }}
        }},
        scales: {{
            x: {{ grid: {{ display: false }} }},
            y: {{
                stacked: true,
                ticks: {{ callback: v => v + '%' }}
            }}
        }}
    }}
}});

// ── Tabell ──
let sortCol = 3;
let sortAsc = false;

function renderTable(data) {{
    const tbody = document.getElementById('tableBody');
    const maxAndel = Math.max(...data.map(r => r.andel_pst));
    tbody.innerHTML = data.map(r => `
        <tr>
            <td>${{r.kommune}}</td>
            <td class="num">${{r.innbyggere.toLocaleString('nb-NO')}}</td>
            <td class="num">${{r.innbyggere_55_pluss.toLocaleString('nb-NO')}}</td>
            <td class="num">
                <div class="andel-bar">
                    <div class="bar" style="width:${{(r.andel_pst/maxAndel*60)}}px"></div>
                    ${{r.andel_pst}}%
                </div>
            </td>
        </tr>
    `).join('');
}}

function sortTable(col) {{
    if (sortCol === col) sortAsc = !sortAsc;
    else {{ sortCol = col; sortAsc = col === 0; }}

    const keys = ['kommune','innbyggere','innbyggere_55_pluss','andel_pst'];
    const key = keys[col];
    const filtered = getFilteredData();
    filtered.sort((a,b) => {{
        let va = a[key], vb = b[key];
        if (typeof va === 'string') return sortAsc ? va.localeCompare(vb) : vb.localeCompare(va);
        return sortAsc ? va - vb : vb - va;
    }});

    document.querySelectorAll('th').forEach((th, i) => {{
        th.classList.toggle('sorted', i === col);
        const arrow = th.querySelector('.sort-arrow');
        if (arrow) arrow.textContent = (i === col) ? (sortAsc ? '↑' : '↓') : '↕';
    }});

    renderTable(filtered);
}}

function getFilteredData() {{
    const q = document.getElementById('tableSearch').value.toLowerCase();
    return dataTabell.filter(r => r.kommune.toLowerCase().includes(q));
}}

function filterTable() {{
    const filtered = getFilteredData();
    const keys = ['kommune','innbyggere','innbyggere_55_pluss','andel_pst'];
    const key = keys[sortCol];
    filtered.sort((a,b) => {{
        let va = a[key], vb = b[key];
        if (typeof va === 'string') return sortAsc ? va.localeCompare(vb) : vb.localeCompare(va);
        return sortAsc ? va - vb : vb - va;
    }});
    renderTable(filtered);
}}

// Init
renderTable(dataTabell);
</script>

</body>
</html>"""

print(f"✓ HTML generert ({len(html):,} tegn)")

## 4. Lagre filen

In [0]:
# Skriv til rotmappen i Pensjon-Lakehouse repoet
output_path = "/Workspace/Users/fredrikvogth@hotmail.com/Pensjon-Lakehouse/index.html"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✓ Dashboard lagret til {output_path}")
print()
print("Neste steg:")
print("  1. Commit og push fra Databricks (eller lokalt via git pull)")
print("  2. GitHub → Settings → Pages → Source: Deploy from branch → / (root)")
print("  3. Besøk: https://fredrikve.github.io/Pensjon-Lakehouse/")

## Alternativ: Vis HTML-en direkte i notebooken

Fjern kommentar under for å se en forhåndsvisning.

In [0]:
# displayHTML(html)